In [1]:
# Necassary library imports
import os
import json
import glob
import time
from datetime import datetime
import logging
import random
from tqdm import tqdm
from pathlib import Path

# Importing Modular File Functions
from api.test import test_api_connection
from config.model import BASE_DIR, MAX_RETRIES, BASE_DELAY, JITTER
from api.gemini_api import call_gemini_api 
from validation.validate import validate_code
from utils.formatting import extract_json_block_from_response, extract_c_code_from_output
from validation.validate_build_command import build_command_looks_cpp

In [2]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
def count_c_files(input_dir: Path) -> int:
    """
    Count the number of .c files in the input directory (recursively).
    
    Args:
        input_dir (Path): The path to the directory to search.
    
    Returns:
        int: Number of .c files found.
    """
    return sum(1 for _ in input_dir.rglob("*.c"))

In [4]:
# Output settings
INPUT_DIR = BASE_DIR / "Files Collection"
VALID_DATA = []
INVALID_DATA = []
VARIATIONS_DATA = []
PROCESSED_EXAMPLES = 0
output_error_file = "raspberry_pi_invalid_data.json"
output_valid_file = "raspberry_pi_validated_data.json"
output_variations_file = "raspberry_pi_variations_data.json"
id = 1
fail_reason = ""
c_file_count = count_c_files(INPUT_DIR)

sucessfully_processed_count = 0
failed_count = 0

In [5]:
generation_instruction_prompt = (
            "IMPORTANT FORMATTING REQUIREMENTS:"
            "Respond with a valid JSON object using the following structure: "
            "{"
            "\"category\": \"Short description of the application\", "
            "\"input\": \"The natural language prompt (Example: Write a C code for Raspberry Pi 5 to read data and log it) -- Donot write words like 'improved/varied version' \", "
            "\"output\": \"The complete C source code as a string\", "
            "\"explanation\": \"Brief explanation of how the code works (2 - 3 lines) \", "
            "\"tags\": \"Comma-separated tags (e.g., Raspberry Pi, C, GPIO)\", "
            "\"file-name\": \"Suggested .c file name\", "
            "\"build-command\": \"The gcc or make command to compile the code\""
            "}. "
            "Avoid unnecessary formatting like markdown or triple backticks."
            "Please donot add anything with the output field of the response json i.e. the code, let it be purely C code. Dont add any Markdown formatting like (```c and ```)"
        )

def save_invalid_data():
        """Save all data to the output file"""
        try:
            with open(output_error_file, 'w') as f:
                json.dump(INVALID_DATA, f, indent=2)
            print(f"Saved {len(INVALID_DATA)} entries to {output_error_file}")
        except Exception as e:
            print(f"Error saving data: {e}")

def save_valid_data():
        """Save all data to the output file"""
        try:
            with open(output_valid_file, 'w') as f:
                json.dump(VALID_DATA, f, indent=2)
            print(f"Saved {len(VALID_DATA)} entries to {output_valid_file}")
        except Exception as e:
            print(f"Error saving data: {e}")

def save_variations_data():
        """Save all data to the output file"""
        try:
            with open(output_variations_file, 'w') as f:
                json.dump(VARIATIONS_DATA, f, indent=2)
            print(f"Saved {len(VARIATIONS_DATA)} entries to {output_variations_file}")
        except Exception as e:
            print(f"Error saving data: {e}")

def is_raspberry_pi_C_code(code, filename):
        """Check if the code is for Raspberry Pi"""
        prompt = f"""
        Analyze this C code and tell me if it is specifically designed for use with a Raspberry Pi.
        Answer with ONLY "YES" or "NO".
        
        Filename: {filename}
        
        Code:
        {code}
        """
        
        response = call_gemini_api(prompt)
        # Check if response contains YES (case insensitive)
        return "YES" in response.upper()

def generate_prompt(code, filename):
        """Generate a detailed prompt describing what the code does"""
        prompt = f"""
        You are an expert Raspberry Pi developer. Analyze this C code for a Raspberry Pi project/task
        and generate a prompt that describes exactly what it does and what task it accomplishes.

        Your prompt should include:
        1. A clear title and description of the project's purpose
        2. Hardware requirements (sensors, pins, connections)
        3. Software dependencies 
        4. Brief explanation of how the code works
        5. Expected outputs and behavior

        Filename: {filename}

        Code:
        {code}

        Strict formatting requirements:
        Example:- Write a C program for Raspberry Pi 5 to read data from a DHT11 sensor connected to GPIO7 and write the readings to a CSV file named 'sensor_log.csv'.

        Always structure your prompt like the example above:
        1). Always start with the keyword 'Write a C program'
        2). Mention target RPI model.
        3). Mention the task and other details
        Keep the prompt in plain english and Donot write a very length prompt, keep it restricted to maximum 2 lines.
        """
        
        return call_gemini_api(prompt)
    
def instruct_to_generate_improved_code(code):
    """Generate an improved variation of the code"""
    prompt = f"""
    You are an expert Raspberry Pi developer. Generated a variation of this C code for a Raspberry Pi project.

    IMPORTANT GENERATION INSTRUCTIONS:
    {generation_instruction_prompt}

    Maintain the same functionality but make improvements for:
    - Better code structure and organization
    - Better error handling
    - Better commenting and documentation
    - Better variable naming
    - Better performance
    
    Original Code:
    {code}

    Output must be a single raw JSON object without any commentary, explanation, or markdown syntax outside the JSON.

    IMPORTANT LAST INSTRUCTIONS:
    {fail_reason}
    """
    
    return prompt
    

def process_directory(directory):
    """Process all C files in a directory"""

    global sucessfully_processed_count
    global failed_count

    logger.info(f"Starting processing Raspberry Pi C code examples")
    
    # Current user info for metadata
    current_user = os.environ.get("USER", "Unknown")
    current_datetime = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
    
    logger.info(f"Current Date and Time (UTC): {current_datetime}")
    logger.info(f"Current User's Login: {current_user}")
    
    # First test the API connection
    if not test_api_connection():
        logger.warning(f"Gemini API unavailable, try again later.")
        return
    
    # Get all C files
    c_files = glob.glob(os.path.join(directory, "**", "*.c"), recursive=True)
    print(f"Found {len(c_files)} C files in {directory}")
    
    
    pbar = tqdm(total=c_file_count, desc="Processing files")
    # Process tasks
    start_time = time.time()
    for file_path in c_files:
        success = process_file(file_path)
        if success:
            sucessfully_processed_count += 1
        else:
            failed_count += 1
        time.sleep(1)
         # Update progress bar
        pbar.update(1)
        pbar.set_description(f"Successfully processed: {sucessfully_processed_count}, Failed: {failed_count}")

        # Calculate and display ETA
        elapsed = time.time() - start_time
        i = sucessfully_processed_count + failed_count
        avg_time_per_item = elapsed / (i + 1)
        remaining_items = len(c_files) - (i + 1)
        eta_seconds = avg_time_per_item * remaining_items
        
        if i % 10 == 0 and i > 0:
            eta_hours = int(eta_seconds // 3600)
            eta_minutes = int((eta_seconds % 3600) // 60)
            logger.info(f"Progress: {i+1}/{len(c_files)} examples. Estimated time remaining: {eta_hours}h {eta_minutes}m")
    
    pbar.close()
    
    # Final report
    end_time = time.time()
    total_time = end_time - start_time
    hours = int(total_time // 3600)
    minutes = int((total_time % 3600) // 60)
    seconds = int(total_time % 60)
    
    logger.info(f"Generation complete: {sucessfully_processed_count} successful, {failed_count} failed")
    logger.info(f"Total execution time: {hours}h {minutes}m {seconds}s")

    logger.info(f"Successfully processed {sucessfully_processed_count} .c files")

def process_file(file_path):
        """Process a single C file"""
        global id
        # Skip if already processed
        if file_path in VALID_DATA or file_path in INVALID_DATA:
            logger.info(f"Skipping already processed file: {file_path}")
            return False
        
        try:
            # Read the file
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            
            if not code.strip():
                logger.warning(f"Empty code file: {file_path}")
                return False
            
            filename = os.path.basename(file_path)
            print(f"Processing {filename}...")
            
            # Check if it's a Raspberry Pi C code
            is_pi_code = is_raspberry_pi_C_code(code, filename)
            
            if not is_pi_code:
                logger.warning(f"Not a Raspberry Pi code: {filename}")
                # Still store the result to avoid reprocessing
                INVALID_DATA.append({
                    "id": str(id),
                    "filename": filename,
                    "is_raspberry_pi_code": False,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "source_code": code,
                    "comment": "Not RPI c code"
                })
                logger.info(f"Saving the code from file: {filename} into the respective json file.")
                save_invalid_data()
                id = id + 1
                return False
            
            # Validating real-world code
            valid, fail_reason = validate_code(code, False)
            
            if not valid:
                logger.warning(f"Not a valid Raspberry Pi code: {filename}")
                # Still store the result to avoid reprocessing
                INVALID_DATA.append({
                    "id": str(id),
                    "filename": filename,
                    "is_raspberry_pi_code": False,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "source_code": code,
                    "comment": f"Not valid RPI c code, reasons: {fail_reason}"
                })
                logger.info(f"Saving the code from file: {filename} into the respective json file.")
                save_invalid_data()
                id = id + 1
                return False
            
            # Generate prompt
            logger.info(f"Generating prompt for {filename}...")
            prompt = generate_prompt(code, filename)

            if prompt:
                logger.info(f"Saving the code in file: {filename} into the respective json file")
                # Still store the result to avoid reprocessing
                VALID_DATA.append({
                    "id": str(id),
                    "filename": filename,
                    "is_raspberry_pi_code": True,
                    "input": prompt,
                    "output": code,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                })
                logger.info(f"Saving the code from file: {filename} into the respective json file.")
                save_valid_data()
                id = id + 1
            
            # Generate improved code
            print(f"Generating variations in code for the file named {filename}...")

            # Try multiple times in case of failure
            for attempt in range(MAX_RETRIES):
                try:
                    # Generate code with slightly higher temperature for creativity
                    code_generation_prompt = instruct_to_generate_improved_code(code)
                    response_text = call_gemini_api(code_generation_prompt, temperature=0.75)

                    if not response_text:
                        logger.warning(f"Empty response from API, retrying ({attempt+1}/{MAX_RETRIES})")
                        time.sleep(BASE_DELAY)
                        continue
                        
                    # Extract and clean the response
                    clean_json = extract_json_block_from_response(response_text)

                    structured_data = json.loads(clean_json)  # Parse cleaned JSON
                    # 🔥 FIX: Unwrap if it's a list
                    if isinstance(structured_data, list):
                        structured_data = structured_data[0]
                    
                    variation_code = structured_data.get("output", "")
                    filtered_code = extract_c_code_from_output(variation_code)
                    build_command = structured_data.get("build-command", "")
                    file_name = structured_data.get("file-name", "")
                    category = structured_data.get("category", "")
                    input = structured_data.get("input", "")
                    explanation = structured_data.get("explanation", "")
                    tags = structured_data.get("tags", "")

                    valid, fail_reason = validate_code(filtered_code)

                    # Validate the code
                    if valid:
                        if not build_command_looks_cpp(build_command):
                            # Save the example with both the AI-generated prompt and the code, and other elements.
                            VARIATIONS_DATA.append({
                                "id": str(id),
                                "is_raspberry_pi_code": True,
                                "category": category,    
                                "input": input,
                                "output": filtered_code,
                                "explanation": explanation,
                                "tags": tags,
                                "file-name":file_name,
                                "build-command": build_command,
                                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            })
                            logger.info(f"Saving the variation for the code from file: {filename} into the respective json file.")
                            logger.info(f"Successfully processed {filename}")
                            save_variations_data()
                            id = id + 1
                            return True
                        else:
                            logger.warning(f"Generated variation has incorrect build command, retrying ({attempt+1}/{MAX_RETRIES})")
                            fail_reason = "Please generated a valid build command for the variation C code"
                    else:
                        logger.warning(f"Generated code failed validation, retrying ({attempt+1}/{MAX_RETRIES})")
                except Exception as e:
                    logger.error(f"Error generating example: {str(e)}")
                
                # Wait before retrying
                time.sleep(BASE_DELAY + random.uniform(0, JITTER))
            
            logger.error(f"Failed to generate valid variation for file: {filename} after {MAX_RETRIES} attempts")
            return False
        
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            return False

In [6]:
if __name__ == "__main__":
    process_directory("C:/Users/Admin/Documents/real-world-data-pipeline/Files Collection")

2025-04-14 16:38:21,353 - INFO - Starting processing Raspberry Pi C code examples
C:\Users\Admin\AppData\Local\Temp\ipykernel_21256\359737157.py:127: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_datetime = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')
2025-04-14 16:38:21,356 - INFO - Current Date and Time (UTC): 2025-04-14 14:38:21
2025-04-14 16:38:21,356 - INFO - Current User's Login: Unknown
2025-04-14 16:38:21,357 - INFO - Testing connection to Gemini API...
2025-04-14 16:38:22,076 - INFO - ✅ API connection successful


Found 196 C files in C:/Users/Admin/Documents/real-world-data-pipeline/Files Collection


Processing files:   0%|          | 0/196 [00:00<?, ?it/s]

Processing file_1.c...


2025-04-14 16:38:22,630 - INFO - Generating prompt for file_1.c...
2025-04-14 16:38:23,244 - INFO - Saving the code in file: file_1.c into the respective json file
2025-04-14 16:38:23,245 - INFO - Saving the code from file: file_1.c into the respective json file.


...

2025-04-14 17:15:14,537 - INFO - Generating prompt for file_99.c...
2025-04-14 17:15:15,374 - INFO - Saving the code in file: file_99.c into the respective json file
2025-04-14 17:15:15,375 - INFO - Saving the code from file: file_99.c into the respective json file.


Saved 176 entries to raspberry_pi_validated_data.json
Generating variations in code for the file named file_99.c...


2025-04-14 17:15:25,434 - INFO - 🔍 Trying to extract JSON block from text...
2025-04-14 17:15:25,434 - INFO - ✅ JSON block found inside ```json wrapper.
2025-04-14 17:15:25,435 - INFO - 🔍 Trying to extract ```c block from output field...
2025-04-14 17:15:25,435 - WARNING - ⚠️ No ```c block found. Using raw output.
2025-04-14 17:15:25,439 - INFO - Saving the variation for the code from file: file_99.c into the respective json file.
2025-04-14 17:15:25,440 - INFO - Successfully processed file_99.c


Saved 174 entries to raspberry_pi_variations_data.json


Successfully processed: 174, Failed: 22: 100%|██████████| 196/196 [37:04<00:00, 11.35s/it]
2025-04-14 17:15:26,456 - INFO - Generation complete: 174 successful, 22 failed
2025-04-14 17:15:26,457 - INFO - Total execution time: 0h 37m 4s
2025-04-14 17:15:26,457 - INFO - Successfully processed 174 .c files
